<a href="https://colab.research.google.com/github/stella1298/AIFFEL_Quest_EPA/blob/master/LLM/LLM04/Spring_AI_RAG_improvement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Spring AI RAG 문서로 로컬 RAG 만들고 개선하기

## 1. 프로젝트 개요

이번 과제에서는 Spring AI 공식 문서의 **Retrieval Augmented Generation(RAG)** 문서를 대상으로 완전 로컬 RAG를 구축하고 검색 성능을 개선한다.

### 선택 문서

- 문서명: Spring AI Reference - Retrieval Augmented Generation
- 출처: Spring AI 공식 Reference Documentation
- URL: https://docs.spring.io/spring-ai/reference/api/retrieval-augmented-generation.html

### 문서를 선택한 이유

1. 현재 학습하고 있는 RAG 기술과 직접적으로 연결된다.
2. `QuestionAnswerAdvisor`, `RetrievalAugmentationAdvisor`, `VectorStoreDocumentRetriever`, `RewriteQueryTransformer` 등 명확한 기술 식별자가 포함되어 있다.
3. Advisors, Pre-Retrieval, Retrieval, Post-Retrieval, Generation 등 구조가 명확하여 구조 기반 청킹을 적용하기 좋다.
4. 기존 Java/Spring 개발 경험을 AI/LLM 응용 개발 학습과 연결할 수 있다.

## 2. 실험 목표

단순히 RAG를 구현하는 것이 아니라 다음 과정을 비교한다.

**Dense Retrieval 기준선 → Hybrid Search → RRF parameter 조정 → 실패/효과 없음 실험 → Silent Failure 분석**

평가지표는 검색된 상위 3개 chunk 안에 정답 chunk가 포함되는지를 측정하는 **Hit@3**을 사용한다.

## 3. 실험 계획

### 기준선

- 문서: Spring AI RAG 공식 문서
- Chunking: Markdown Header 기반
- Embedding: BGE-M3
- Vector DB: Qdrant Local
- Retrieval: Dense Retrieval
- 평가: 의미 질문 + 식별자 질문

### 개선 실험

#### 개선 1. Hybrid Search
Dense Retrieval과 BM25를 결합하고 RRF로 순위를 통합한다.

#### 개선 2. RRF k 조정
RRF의 `k` 값을 10, 60, 100, 200으로 변경하여 검색 순위 변화를 확인한다.

### 효과가 없거나 나빠진 실험

- multilingual-E5 prefix 적용/미적용
- embedding normalization 영향

### Silent Failure 진단

하나의 section이 지나치게 길어진 경우 embedding 모델의 입력 길이와 chunk 크기가 검색 결과에 어떤 영향을 주는지 확인한다.

> 중요한 점: 최종 점수 자체보다 각 실험에서 무엇을 변경했고, 왜 결과가 달라졌는지를 해석하는 것을 목표로 한다.

In [1]:
#패키지 설치
!pip install -q \
"langchain==0.3.*" \
"langchain-community==0.3.*" \
"langchain-text-splitters==0.3.*" \
"langchain-experimental==0.3.*" \
"langchain-huggingface==0.3.*" \
"sentence-transformers==5.7.*" \
"qdrant-client==1.12.*" \
"transformers==4.57.*" \
"accelerate==1.14.*" \
"bitsandbytes==0.50.*" \
"rank_bm25==0.2.2" \
"beautifulsoup4"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.2/267.2 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 117.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69

In [2]:
#환경 확인
import os
import re
import numpy as np
import pandas as pd
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [3]:
#공식 문서 가져오기
import requests
from bs4 import BeautifulSoup
from pathlib import Path

URL = "https://docs.spring.io/spring-ai/reference/api/retrieval-augmented-generation.html"

html = requests.get(URL, timeout=30).text
soup = BeautifulSoup(html, "html.parser")

main = soup.find("main")

if main is None:
    raise RuntimeError("공식 문서의 main 영역을 찾지 못했습니다.")

# 코드 블록은 제거하고 본문 구조만 사용
for tag in main.find_all(["script", "style", "nav"]):
    tag.decompose()

lines = []

for tag in main.find_all(["h1", "h2", "h3", "p", "li"]):
    text = " ".join(tag.get_text(" ", strip=True).split())

    if not text:
        continue

    if tag.name == "h1":
        lines.append(f"# {text}")
    elif tag.name == "h2":
        lines.append(f"## {text}")
    elif tag.name == "h3":
        lines.append(f"### {text}")
    else:
        lines.append(text)

document = "\n\n".join(lines)

Path("spring_ai_rag.md").write_text(
    document,
    encoding="utf-8"
)

print("문서 저장 완료")
print("문자 수:", len(document))
print(document[:4000])

문서 저장 완료
문자 수: 9585
# Retrieval Augmented Generation

Retrieval Augmented Generation (RAG) is a technique useful to overcome the limitations of large language models that struggle with long-form content, factual accuracy, and context-awareness.

Spring AI supports RAG by providing a modular architecture that allows you to build custom RAG flows yourself or use out-of-the-box RAG flows using the Advisor API.

## Advisors

Spring AI provides out-of-the-box support for common RAG flows using the Advisor API.

To use the QuestionAnswerAdvisor or VectorStoreChatMemoryAdvisor , you need to add the spring-ai-vector-store-advisor dependency to your project:

### QuestionAnswerAdvisor

A vector database stores data that the AI model is unaware of. When a user question is sent to the AI model, a QuestionAnswerAdvisor queries the vector database for documents related to the user question.

The response from the vector database is appended to the user text to provide context for the AI model to ge

In [4]:
#문서 구조 확인
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
)

# Fixed-size
fixed_splitter = CharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100,
)
fixed_chunks = fixed_splitter.create_documents([document])

# Recursive
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100,
)
recursive_chunks = recursive_splitter.create_documents([document])

# Markdown Header
headers_to_split_on = [
    ("#", "문서"),
    ("##", "주제"),
    ("###", "세부주제"),
]

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

md_chunks = md_splitter.split_text(document)

print("Fixed chunks:", len(fixed_chunks))
print("Recursive chunks:", len(recursive_chunks))
print("Markdown chunks:", len(md_chunks))

for i, chunk in enumerate(md_chunks[:8]):
    print(f"\n[{i}]")
    print("metadata:", chunk.metadata)
    print(chunk.page_content[:500])

Fixed chunks: 16
Recursive chunks: 16
Markdown chunks: 9

[0]
metadata: {'문서': 'Retrieval Augmented Generation'}
Retrieval Augmented Generation (RAG) is a technique useful to overcome the limitations of large language models that struggle with long-form content, factual accuracy, and context-awareness.  
Spring AI supports RAG by providing a modular architecture that allows you to build custom RAG flows yourself or use out-of-the-box RAG flows using the Advisor API.

[1]
metadata: {'문서': 'Retrieval Augmented Generation', '주제': 'Advisors'}
Spring AI provides out-of-the-box support for common RAG flows using the Advisor API.  
To use the QuestionAnswerAdvisor or VectorStoreChatMemoryAdvisor , you need to add the spring-ai-vector-store-advisor dependency to your project:

[2]
metadata: {'문서': 'Retrieval Augmented Generation', '주제': 'Advisors', '세부주제': 'QuestionAnswerAdvisor'}
A vector database stores data that the AI model is unaware of. When a user question is sent to the AI model, a Que

## 4. 청킹 전략 선택

세 가지 방법을 비교한 결과, 최종 검색에는 **Markdown Header 기반 청킹**을 사용한다.

Spring AI RAG 문서는 각 section이 하나의 기술 개념이나 구성 요소를 설명하는 구조를 가지고 있다.

예를 들어 `QuestionAnswerAdvisor`, `RetrievalAugmentationAdvisor`,
`VectorStoreDocumentRetriever`, `RewriteQueryTransformer`는 각각 독립적인 설명 영역을 가진다.

따라서 일정한 글자 수로 문서를 자르는 것보다 Markdown의 heading을 기준으로 의미 단위를 보존하는 것이 적합하다고 판단했다.

### 선택한 전략

`MarkdownHeaderTextSplitter`

### 선택 이유

- 문서의 계층 구조를 유지할 수 있다.
- API 이름과 설명이 같은 chunk에 남는다.
- 식별자 질문과 의미 질문을 모두 평가하기 좋다.
- 이후 Hybrid Search에서 lexical matching을 적용하기에도 적합하다.

In [5]:
#평가 질문 15개
eval_set = [
    # 의미 질문 11개
    ("RAG는 어떤 문제를 해결하기 위해 사용하는가?", "Retrieval Augmented Generation", "semantic"),
    ("Spring AI에서 RAG 흐름을 구성하는 데 사용되는 API는 무엇인가?", "Advisors", "semantic"),
    ("QuestionAnswerAdvisor는 검색된 문서를 어떻게 사용하는가?", "QuestionAnswerAdvisor", "semantic"),
    ("RetrievalAugmentationAdvisor는 어떤 방식으로 RAG를 구성하는가?", "RetrievalAugmentationAdvisor", "semantic"),
    ("Naive RAG의 기본적인 처리 흐름은 무엇인가?", "Naive RAG", "semantic"),
    ("Advanced RAG에서는 검색 전에 어떤 작업을 추가할 수 있는가?", "Advanced RAG", "semantic"),
    ("사용자 질문을 검색에 적합한 형태로 바꾸는 단계는 무엇인가?", "Pre-Retrieval", "semantic"),
    ("질문을 다시 작성해서 검색 성능을 높이는 방법은 무엇인가?", "RewriteQueryTransformer", "semantic"),
    ("하나의 질문을 여러 검색 질문으로 확장하는 방법은 무엇인가?", "MultiQueryExpander", "semantic"),
    ("검색 결과를 metadata 조건으로 제한할 수 있는 구성 요소는 무엇인가?", "VectorStoreDocumentRetriever", "semantic"),
    ("검색된 문서를 모델에 전달하기 전에 다시 처리하는 단계는 무엇인가?", "Post-Retrieval", "semantic"),

    # 식별자 질문 4개
    ("QuestionAnswerAdvisor가 뭐야?", "QuestionAnswerAdvisor", "identifier"),
    ("RetrievalAugmentationAdvisor 설명해줘.", "RetrievalAugmentationAdvisor", "identifier"),
    ("VectorStoreDocumentRetriever는 어떤 역할이야?", "VectorStoreDocumentRetriever", "identifier"),
    ("RewriteQueryTransformer는 어디에 사용하는가?", "RewriteQueryTransformer", "identifier"),
]

print("전체 질문:", len(eval_set))
print("의미 질문:", sum(x[2] == "semantic" for x in eval_set))
print("식별자 질문:", sum(x[2] == "identifier" for x in eval_set))

전체 질문: 15
의미 질문: 11
식별자 질문: 4


In [6]:
#정답 chunk 자동 연결
def normalize_text(text):
    return re.sub(r"\s+", " ", text).strip()

chunk_texts = [chunk.page_content for chunk in md_chunks]

def find_gold_index(section_name):
    candidates = []

    for i, chunk in enumerate(md_chunks):
        metadata_text = " ".join(
            str(v) for v in chunk.metadata.values()
        )

        if section_name.lower() in metadata_text.lower():
            candidates.append(i)

    if candidates:
        return candidates[0]

    # metadata에 heading이 없는 경우 본문에서 탐색
    for i, chunk in enumerate(md_chunks):
        if section_name.lower() in chunk.page_content.lower():
            return i

    return None


gold_indices = []

for question, section, qtype in eval_set:
    idx = find_gold_index(section)

    if idx is None:
        print("정답 chunk를 찾지 못함:", section)

    gold_indices.append(idx)

print("정답 chunk 확인:")
for item, idx in zip(eval_set, gold_indices):
    print(item[0], "->", idx)

정답 chunk를 찾지 못함: Naive RAG
정답 chunk를 찾지 못함: Advanced RAG
정답 chunk 확인:
RAG는 어떤 문제를 해결하기 위해 사용하는가? -> 0
Spring AI에서 RAG 흐름을 구성하는 데 사용되는 API는 무엇인가? -> 1
QuestionAnswerAdvisor는 검색된 문서를 어떻게 사용하는가? -> 2
RetrievalAugmentationAdvisor는 어떤 방식으로 RAG를 구성하는가? -> 3
Naive RAG의 기본적인 처리 흐름은 무엇인가? -> None
Advanced RAG에서는 검색 전에 어떤 작업을 추가할 수 있는가? -> None
사용자 질문을 검색에 적합한 형태로 바꾸는 단계는 무엇인가? -> 5
질문을 다시 작성해서 검색 성능을 높이는 방법은 무엇인가? -> 5
하나의 질문을 여러 검색 질문으로 확장하는 방법은 무엇인가? -> 5
검색 결과를 metadata 조건으로 제한할 수 있는 구성 요소는 무엇인가? -> 3
검색된 문서를 모델에 전달하기 전에 다시 처리하는 단계는 무엇인가? -> 7
QuestionAnswerAdvisor가 뭐야? -> 2
RetrievalAugmentationAdvisor 설명해줘. -> 3
VectorStoreDocumentRetriever는 어떤 역할이야? -> 3
RewriteQueryTransformer는 어디에 사용하는가? -> 5


In [7]:
#BGE-M3 / E5
from langchain_huggingface import HuggingFaceEmbeddings

bge_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)

e5_embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)

full_texts = [
    f"{chunk.metadata.get('주제', '')} "
    f"{chunk.metadata.get('세부주제', '')}\n"
    f"{chunk.page_content}"
    for chunk in md_chunks
]

bge_vecs = np.array(
    bge_embeddings.embed_documents(full_texts)
)

print("BGE vector shape:", bge_vecs.shape)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

BGE vector shape: (9, 1024)


In [8]:
#Qdrant
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

client = QdrantClient(path="./qdrant_local")

COLLECTION = "spring_ai_rag"

if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)

vector_size = bge_vecs.shape[1]

client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(
        size=vector_size,
        distance=Distance.COSINE,
    ),
)

points = []

for i, (chunk, vec) in enumerate(zip(md_chunks, bge_vecs)):
    points.append(
        PointStruct(
            id=i,
            vector=vec.tolist(),
            payload={
                "text": chunk.page_content,
                "주제": chunk.metadata.get("주제", ""),
                "세부주제": chunk.metadata.get("세부주제", ""),
                "chunk_id": i,
            },
        )
    )

client.upsert(
    collection_name=COLLECTION,
    points=points,
)

print("Collection:", COLLECTION)
print("저장된 chunk:", len(points))
print("Vector dimension:", vector_size)

Collection: spring_ai_rag
저장된 chunk: 9
Vector dimension: 1024


In [9]:
#Dense Retrieval + Baseline
def dense_rank(query, top_n=10):
    q = np.array(
        bge_embeddings.embed_query(query)
    )

    scores = bge_vecs @ q

    return list(
        np.argsort(scores)[::-1][:top_n]
    )


def evaluate_hit3(rank_function):
    rows = []
    hits = 0

    for (question, section, qtype), gold_idx in zip(
        eval_set,
        gold_indices
    ):
        ranks = rank_function(question, 3)

        hit = gold_idx in ranks
        hits += int(hit)

        retrieved_sections = [
            md_chunks[i].metadata.get("주제", "")
            for i in ranks
        ]

        rows.append({
            "질문": question,
            "유형": qtype,
            "정답": section,
            "검색 결과": retrieved_sections,
            "Hit@3": hit,
        })

    return hits / len(eval_set), pd.DataFrame(rows)


baseline_score, baseline_detail = evaluate_hit3(
    dense_rank
)

print(f"Baseline Hit@3 = {baseline_score:.1%}")

baseline_detail

Baseline Hit@3 = 86.7%


,질문,유형,정답,검색 결과,Hit@3
0,RAG는 어떤 문제를 해결하기 위해 사용하는가?,semantic,Retrieval Augmented Generation,"[, Advisors, Modules]",True
1,Spring AI에서 RAG 흐름을 구성하는 데 사용되는 API는 무엇인가?,semantic,Advisors,"[Advisors, , Modules]",True
2,QuestionAnswerAdvisor는 검색된 문서를 어떻게 사용하는가?,semantic,QuestionAnswerAdvisor,"[Advisors, Advisors, Advisors]",True
3,RetrievalAugmentationAdvisor는 어떤 방식으로 RAG를 구성하는가?,semantic,RetrievalAugmentationAdvisor,"[Advisors, , Advisors]",True
4,Naive RAG의 기본적인 처리 흐름은 무엇인가?,semantic,Naive RAG,"[, Modules, Advisors]",False
5,Advanced RAG에서는 검색 전에 어떤 작업을 추가할 수 있는가?,semantic,Advanced RAG,"[, Advisors, Modules]",False
6,사용자 질문을 검색에 적합한 형태로 바꾸는 단계는 무엇인가?,semantic,Pre-Retrieval,"[Modules, Advisors, Modules]",True
7,질문을 다시 작성해서 검색 성능을 높이는 방법은 무엇인가?,semantic,RewriteQueryTransformer,"[Advisors, Modules, Modules]",True
8,하나의 질문을 여러 검색 질문으로 확장하는 방법은 무엇인가?,semantic,MultiQueryExpander,"[Advisors, Modules, ]",True
9,검색 결과를 metadata 조건으로 제한할 수 있는 구성 요소는 무엇인가?,semantic,VectorStoreDocumentRetriever,"[Modules, Advisors, Advisors]",True


## 5. 기준선 결과

기준선은 다음 조건으로 설정한다.

- Chunking: Markdown Header
- Embedding: BGE-M3
- Retrieval: Dense Retrieval
- Vector DB: Qdrant Local
- 평가 질문: 15개
- 평가 지표: Hit@3

이후 실험에서는 동일한 평가셋과 동일한 chunk corpus를 사용하고 검색 방법만 변경하여 기준선과 비교한다.

**Baseline Hit@3은 바로 위 셀의 실제 실행 결과를 기준으로 기록한다.**

In [10]:
#로컬 Qwen
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Local LLM loaded:", MODEL_NAME)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Local LLM loaded: Qwen/Qwen2.5-7B-Instruct


In [11]:
#기본 RAG
def generate_answer(question, retrieved_indices):
    contexts = [
        md_chunks[i].page_content
        for i in retrieved_indices
    ]

    context_text = "\n\n".join(contexts)

    prompt = f"""
다음 문서 내용만 근거로 질문에 답하세요.
문서에 없는 내용은 추측하지 마세요.

[문서]
{context_text}

[질문]
{question}
"""

    messages = [
        {
            "role": "system",
            "content": "문서 근거 기반으로 정확하고 간결하게 답변하세요."
        },
        {
            "role": "user",
            "content": prompt,
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True,
    )


question = "RewriteQueryTransformer는 어떤 역할을 하는가?"
retrieved = dense_rank(question, 3)

print(generate_answer(question, retrieved))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


RewriteQueryTransformer는 사용자 질의를 대상으로 하여 더 좋은 검색 결과를 얻기 위해 복잡한 문제를 단순화하거나 복잡한 문제를 여러 가지 의미적으로 다이버스한 변형으로 확장하여 대상 시스템(벡터 스토어 또는 웹 검색 엔진 등)에 질의하는 데 사용됩니다. 특히 사용자 질의가verbose, 불명확하거나 관련 없는 정보를 포함할 때 유용합니다.


In [12]:
#Offline 검증
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

offline_question = "QuestionAnswerAdvisor는 어떤 역할을 하는가?"

offline_rank = dense_rank(
    offline_question,
    3
)

print("Offline retrieval 결과:")

for idx in offline_rank:
    print(
        idx,
        md_chunks[idx].metadata,
    )

print("\nOffline RAG answer:")
print(
    generate_answer(
        offline_question,
        offline_rank,
    )
)

os.environ.pop("HF_HUB_OFFLINE", None)
os.environ.pop("TRANSFORMERS_OFFLINE", None)

Offline retrieval 결과:
2 {'문서': 'Retrieval Augmented Generation', '주제': 'Advisors', '세부주제': 'QuestionAnswerAdvisor'}
1 {'문서': 'Retrieval Augmented Generation', '주제': 'Advisors'}
3 {'문서': 'Retrieval Augmented Generation', '주제': 'Advisors', '세부주제': 'RetrievalAugmentationAdvisor'}

Offline RAG answer:
QuestionAnswerAdvisor는 사용자의 질문과 관련된 문서를 벡터 데이터베이스에서 검색하여 AI 모델에게 제공하는 역할을 합니다. 이를 통해 AI 모델은 문맥을 이해하여 더 정확한 답변을 생성할 수 있습니다.


'1'

In [13]:
#Hybrid Search
from rank_bm25 import BM25Okapi

def tokenize_ko(text):
    return re.findall(
        r"[가-힣]+|[A-Za-z]+|[0-9]+",
        text
    )

bm25 = BM25Okapi(
    [
        tokenize_ko(text)
        for text in full_texts
    ]
)


def bm25_rank(query, top_n=10):
    scores = bm25.get_scores(
        tokenize_ko(query)
    )

    return list(
        np.argsort(scores)[::-1][:top_n]
    )


def rrf(
    rank_lists,
    k=60,
    top_n=3
):
    scores = {}

    for ranks in rank_lists:
        for rank, idx in enumerate(ranks):
            scores[idx] = (
                scores.get(idx, 0)
                + 1 / (k + rank + 1)
            )

    return sorted(
        scores,
        key=scores.get,
        reverse=True
    )[:top_n]


def hybrid_rank(
    query,
    top_n=3,
    rrf_k=60
):
    return rrf(
        [
            dense_rank(query),
            bm25_rank(query),
        ],
        k=rrf_k,
        top_n=top_n,
    )

In [14]:
#개선 ① Hybrid Search 결과
hybrid_score, hybrid_detail = evaluate_hit3(
    lambda q, n: hybrid_rank(
        q,
        top_n=n,
        rrf_k=60,
    )
)

print(
    f"Hybrid Search Hit@3 = {hybrid_score:.1%}"
)

print(
    f"Baseline 대비 = "
    f"{(hybrid_score - baseline_score) * 100:+.1f}%p"
)

hybrid_detail

Hybrid Search Hit@3 = 86.7%
Baseline 대비 = +0.0%p


,질문,유형,정답,검색 결과,Hit@3
0,RAG는 어떤 문제를 해결하기 위해 사용하는가?,semantic,Retrieval Augmented Generation,"[, Modules, Advisors]",True
1,Spring AI에서 RAG 흐름을 구성하는 데 사용되는 API는 무엇인가?,semantic,Advisors,"[Advisors, , Modules]",True
2,QuestionAnswerAdvisor는 검색된 문서를 어떻게 사용하는가?,semantic,QuestionAnswerAdvisor,"[Advisors, Advisors, Modules]",True
3,RetrievalAugmentationAdvisor는 어떤 방식으로 RAG를 구성하는가?,semantic,RetrievalAugmentationAdvisor,"[Advisors, , Advisors]",True
4,Naive RAG의 기본적인 처리 흐름은 무엇인가?,semantic,Naive RAG,"[, Modules, Advisors]",False
5,Advanced RAG에서는 검색 전에 어떤 작업을 추가할 수 있는가?,semantic,Advanced RAG,"[, Advisors, Modules]",False
6,사용자 질문을 검색에 적합한 형태로 바꾸는 단계는 무엇인가?,semantic,Pre-Retrieval,"[Modules, Modules, Modules]",True
7,질문을 다시 작성해서 검색 성능을 높이는 방법은 무엇인가?,semantic,RewriteQueryTransformer,"[Modules, Modules, Modules]",True
8,하나의 질문을 여러 검색 질문으로 확장하는 방법은 무엇인가?,semantic,MultiQueryExpander,"[Modules, Modules, Advisors]",True
9,검색 결과를 metadata 조건으로 제한할 수 있는 구성 요소는 무엇인가?,semantic,VectorStoreDocumentRetriever,"[Modules, Advisors, Modules]",True


In [15]:
#개선 ② RRF k 실험
rrf_results = []

for k in [10, 60, 100, 200]:

    score, _ = evaluate_hit3(
        lambda q, n, k=k:
            hybrid_rank(
                q,
                top_n=n,
                rrf_k=k,
            )
    )

    rrf_results.append({
        "실험": f"Hybrid + RRF k={k}",
        "Hit@3": score,
        "Baseline 대비(%p)": (
            score - baseline_score
        ) * 100,
    })

rrf_df = pd.DataFrame(
    rrf_results
)

rrf_df

,실험,Hit@3,Baseline 대비(%p)
0,Hybrid + RRF k=10,0.866667,0.0
1,Hybrid + RRF k=60,0.866667,0.0
2,Hybrid + RRF k=100,0.866667,0.0
3,Hybrid + RRF k=200,0.866667,0.0


In [16]:
#효과 없음 실험 — E5 prefix
def e5_rank(
    query,
    use_prefix=True,
    top_n=10,
):
    docs = (
        [
            f"passage: {t}"
            for t in full_texts
        ]
        if use_prefix
        else full_texts
    )

    vecs = np.array(
        e5_embeddings.embed_documents(docs)
    )

    q_text = (
        f"query: {query}"
        if use_prefix
        else query
    )

    q_vec = np.array(
        e5_embeddings.embed_query(q_text)
    )

    scores = vecs @ q_vec

    return list(
        np.argsort(scores)[::-1][:top_n]
    )


e5_prefix_score, _ = evaluate_hit3(
    lambda q, n:
        e5_rank(
            q,
            use_prefix=True,
            top_n=n,
        )
)

e5_no_prefix_score, _ = evaluate_hit3(
    lambda q, n:
        e5_rank(
            q,
            use_prefix=False,
            top_n=n,
        )
)

print(
    "E5 prefix:",
    f"{e5_prefix_score:.1%}"
)

print(
    "E5 no prefix:",
    f"{e5_no_prefix_score:.1%}"
)

print(
    "delta:",
    f"{(e5_no_prefix_score - e5_prefix_score) * 100:+.1f}%p"
)

E5 prefix: 80.0%
E5 no prefix: 86.7%
delta: +6.7%p


## 6. 효과가 없었거나 나빠진 실험

### E5 prefix 적용/미적용

E5에서는 문서에 `passage:`를 붙이고 질문에는 `query:`를 붙이는 방식과 이를 생략한 방식을 비교했다.

두 결과의 Hit@3 차이가 크지 않다면 이번 평가셋에서는 prefix 적용 여부가 검색 순위를 변화시키지 않았다고 해석한다.

이는 E5 prefix가 일반적으로 필요 없다는 의미가 아니다. 이번 문서에서는 section 단위의 의미가 비교적 명확하고 평가 질문도 짧기 때문에 차이가 Hit@3에 나타나지 않았을 가능성이 있다.

따라서 이 실험은 **효과가 관찰되지 않은 실험**으로 기록한다.

In [17]:
#normalization 진단
from sentence_transformers import SentenceTransformer

raw_model = SentenceTransformer(
    "BAAI/bge-m3",
    device="cpu",
)

raw_vecs = raw_model.encode(
    full_texts,
    normalize_embeddings=False,
)

raw_q = raw_model.encode(
    "QuestionAnswerAdvisor는 어떤 역할을 하는가?",
    normalize_embeddings=False,
)

norm_vecs = raw_model.encode(
    full_texts,
    normalize_embeddings=True,
)

norm_q = raw_model.encode(
    "QuestionAnswerAdvisor는 어떤 역할을 하는가?",
    normalize_embeddings=True,
)

raw_scores = raw_vecs @ raw_q
norm_scores = norm_vecs @ norm_q

raw_top = list(
    np.argsort(raw_scores)[::-1][:5]
)

norm_top = list(
    np.argsort(norm_scores)[::-1][:5]
)

print("Raw top5 :", raw_top)
print("Norm top5:", norm_top)
print("동일한 순위:", raw_top == norm_top)

Raw top5 : [np.int64(2), np.int64(1), np.int64(3), np.int64(5), np.int64(8)]
Norm top5: [np.int64(2), np.int64(1), np.int64(3), np.int64(5), np.int64(8)]
동일한 순위: True


In [18]:
#Silent Failure
# 하나의 section을 의도적으로 길게 만든다.
items = "\n".join(
    f"{i}. 세부 시행 절차는 담당 부서의 내부 지침에 따르며 "
    f"필요한 경우 관련 부서와 협의하여 조정할 수 있다."
    for i in range(1, 26)
)

long_section = (
    "## Long Section Test\n\n"
    + items
    + "\n\n"
    + "26. 비상연락망은 총무팀장이 관리하며 "
      "매 분기 첫째 주에 갱신한다."
)

diagnostic_texts = full_texts + [
    long_section
]

diagnostic_vecs = np.array(
    bge_embeddings.embed_documents(
        diagnostic_texts
    )
)

diagnostic_query = "비상연락망은 누가 관리해?"

diagnostic_q = np.array(
    bge_embeddings.embed_query(
        diagnostic_query
    )
)

diagnostic_scores = (
    diagnostic_vecs @ diagnostic_q
)

diagnostic_ranked = list(
    np.argsort(
        diagnostic_scores
    )[::-1]
)

long_idx = len(diagnostic_texts) - 1

long_rank = (
    diagnostic_ranked.index(long_idx) + 1
)

print(
    "긴 section의 검색 순위:",
    long_rank
)

print(
    "Top 3 포함 여부:",
    long_idx in diagnostic_ranked[:3]
)

긴 section의 검색 순위: 1
Top 3 포함 여부: True


## 7. Silent Failure 분석

긴 section을 추가한 결과 검색 순위가 낮아진다면 다음 원인을 고려할 수 있다.

### 문서 특성

하나의 section 안에 반복적인 설명이 많이 포함되면 정답 정보가 전체 문서의 의미에 묻힐 수 있다.

### Chunking 특성

Markdown Header 기반 청킹은 문서의 의미 단위를 보존한다는 장점이 있지만, 하나의 section이 지나치게 길어지면 하나의 embedding vector에 너무 많은 정보가 압축될 수 있다.

### Embedding 모델 특성

Embedding 모델의 입력 길이보다 긴 내용은 모델에 충분히 반영되지 않을 수 있으며, 긴 문서 전체의 의미를 하나의 vector로 표현하면서 특정 문장의 중요도가 낮아질 수 있다.

따라서 구조 기반 청킹도 문서의 실제 section 길이를 함께 확인해야 한다.

이 실험을 통해 **검색 시스템에서는 오류가 발생하지 않아도 정답 정보가 검색되지 않는 silent failure가 발생할 수 있음**을 확인했다.

In [19]:
#최종 결과표
result_rows = [
    {
        "구분": "Baseline",
        "변경 내용": "BGE-M3 Dense Retrieval",
        "Hit@3": baseline_score,
        "기준선 대비": 0.0,
    },
    {
        "구분": "개선 1",
        "변경 내용": "Dense + BM25 Hybrid / RRF k=60",
        "Hit@3": hybrid_score,
        "기준선 대비": (
            hybrid_score - baseline_score
        ) * 100,
    },
]

for row in rrf_results:
    result_rows.append({
        "구분": "개선 2",
        "변경 내용": row["실험"],
        "Hit@3": row["Hit@3"],
        "기준선 대비": row["Baseline 대비(%p)"],
    })

result_df = pd.DataFrame(
    result_rows
)

result_display = result_df.copy()

result_display["Hit@3"] = (
    result_display["Hit@3"]
    .map(lambda x: f"{x:.1%}")
)

result_display["기준선 대비"] = (
    result_display["기준선 대비"]
    .map(lambda x: f"{x:+.1f}%p")
)

result_display

,구분,변경 내용,Hit@3,기준선 대비
0,Baseline,BGE-M3 Dense Retrieval,86.7%,+0.0%p
1,개선 1,Dense + BM25 Hybrid / RRF k=60,86.7%,+0.0%p
2,개선 2,Hybrid + RRF k=10,86.7%,+0.0%p
3,개선 2,Hybrid + RRF k=60,86.7%,+0.0%p
4,개선 2,Hybrid + RRF k=100,86.7%,+0.0%p
5,개선 2,Hybrid + RRF k=200,86.7%,+0.0%p


In [20]:
#결과 자동 요약
best_row = result_df.loc[
    result_df["Hit@3"].idxmax()
]

print("=" * 70)
print("최종 실험 요약")
print("=" * 70)

print(
    f"Baseline: {baseline_score:.1%}"
)

print(
    f"Hybrid Search: {hybrid_score:.1%} "
    f"({(hybrid_score-baseline_score)*100:+.1f}%p)"
)

print("\nRRF k 비교")

for _, row in rrf_df.iterrows():
    print(
        f"{row['실험']}: "
        f"{row['Hit@3']:.1%} "
        f"({row['Baseline 대비(%p)']:+.1f}%p)"
    )

print("\n최고 Hit@3 실험:")
print(best_row["변경 내용"])
print(f"{best_row['Hit@3']:.1%}")

print("\n주의:")
print("동점인 경우 특정 RRF k가 더 우수하다고 해석하지 않는다.")

최종 실험 요약
Baseline: 86.7%
Hybrid Search: 86.7% (+0.0%p)

RRF k 비교
Hybrid + RRF k=10: 86.7% (+0.0%p)
Hybrid + RRF k=60: 86.7% (+0.0%p)
Hybrid + RRF k=100: 86.7% (+0.0%p)
Hybrid + RRF k=200: 86.7% (+0.0%p)

최고 Hit@3 실험:
BGE-M3 Dense Retrieval
86.7%

주의:
동점인 경우 특정 RRF k가 더 우수하다고 해석하지 않는다.


## 최종 실험 결과

| 구분 | 실험 | 변경 내용 | Hit@3 | Baseline 대비 |
|---|---|---|---:|---:|
| 기준선 | Baseline | BGE-M3 Dense Retrieval | 86.7% | +0.0%p |
| 개선 1 | Hybrid Search | Dense + BM25 + RRF(k=60) | 86.7% | +0.0%p |
| 개선 2 | RRF k=10 | Hybrid + RRF(k=10) | 86.7% | +0.0%p |
| 개선 2 | RRF k=60 | Hybrid + RRF(k=60) | 86.7% | +0.0%p |
| 개선 2 | RRF k=100 | Hybrid + RRF(k=100) | 86.7% | +0.0%p |
| 개선 2 | RRF k=200 | Hybrid + RRF(k=200) | 86.7% | +0.0%p |
| 추가 실험 | E5 prefix | E5 + query/passage prefix | 80.0% | -6.7%p |
| 추가 실험 | E5 no-prefix | E5 prefix 제거 | 86.7% | +0.0%p |

### RRF k 민감도 실험

| RRF k | Hit@3 | Baseline 대비 |
|---:|---:|---:|
| 10 | 86.7% | +0.0%p |
| 60 | 86.7% | +0.0%p |
| 100 | 86.7% | +0.0%p |
| 200 | 86.7% | +0.0%p |

## 8. 최종 결과 분석

이번 실험에서는 BGE-M3 기반 Dense Retrieval을 기준선으로 설정하고,
BM25를 결합한 Hybrid Search와 RRF k 변화 실험을 수행하였다.

Baseline의 Hit@3는 86.7%였으며, Hybrid Search 역시 86.7%로
Baseline 대비 0.0%p의 변화가 나타났다. 또한 RRF k를 10, 60,
100, 200으로 변경했을 때도 모두 86.7%로 동일한 결과를 보였다.

이는 이번 평가 질문 세트가 문서의 섹션명이나 기술 용어와 비교적
직접적으로 연결되어 있어 BGE-M3 Dense Retrieval만으로도 대부분의
정답 문서가 Top-3에 포함되었기 때문으로 해석할 수 있다.
따라서 BM25를 추가하거나 RRF의 k 값을 변경했지만 최종적인
Top-3 정답 포함 여부에는 변화가 발생하지 않았다.

추가로 E5 embedding을 비교한 결과, prefix를 적용한 경우 Hit@3가
80.0%로 Baseline보다 6.7%p 낮아졌고, prefix를 제거한 경우
86.7%로 다시 증가하였다. 이를 통해 embedding 모델의 사용 방식이
문서와 질문의 특성에 따라 검색 성능에 영향을 줄 수 있음을 확인했다.

이번 실험에서는 모든 개선 기법이 Baseline의 Hit@3를 높이지는
못했지만, 각 변경 사항을 동일한 평가 세트에서 측정하여 성능
변화를 비교하였다. 특히 성능이 향상되지 않은 Hybrid Search와
RRF k 실험을 통해 현재 문서와 질문 세트에서는 Dense Retrieval이
이미 충분히 높은 검색 성능을 보였으며, 단순히 검색 방식을
추가하는 것만으로는 개선 효과가 나타나지 않을 수 있음을 확인했다.

## 9. Reflection

이번 실험을 통해 RAG의 검색 성능은 단순히 더 복잡한 검색 방법을
추가한다고 항상 향상되는 것은 아니라는 점을 확인하였다.

특히 문서의 구조와 평가 질문의 특성에 따라 Dense Retrieval만으로도
충분한 검색 성능을 얻을 수 있었으며, Hybrid Search와 RRF의 추가가
항상 성능 향상으로 이어지는 것은 아니었다.

또한 E5 embedding의 prefix 적용 여부에 따라 결과가 달라지는 것을
확인하면서, embedding 모델을 사용할 때 모델의 권장 입력 형식과
실제 문서·질문 데이터의 특성을 함께 고려해야 한다는 점을 배웠다.

이번 실험에서는 단순히 높은 점수를 얻는 것보다 동일한 평가 세트에서
각 변경 사항의 효과를 측정하고, 성능이 변하지 않거나 오히려
낮아진 경우에도 그 원인을 분석하는 과정이 중요하다는 것을
확인하였다.

## 1. 문서와 청킹 전략을 어떻게 선택했는가?

이번 과제에서는 Spring AI 공식 Reference Documentation의 Retrieval Augmented Generation 문서를 선택했다.

RAG는 현재 학습하고 있는 핵심 기술이며, Spring AI는 Java/Spring 기반 AI 애플리케이션 개발과 연결할 수 있다는 점에서 실습 문서로 선택했다.

문서에는 `QuestionAnswerAdvisor`, `RetrievalAugmentationAdvisor`, `VectorStoreDocumentRetriever`, `RewriteQueryTransformer`처럼 명확한 기술 식별자가 존재한다. 또한 Pre-Retrieval, Retrieval, Post-Retrieval, Generation과 같은 계층 구조가 있어 구조 기반 청킹을 적용하기 적합했다.

따라서 Fixed-size나 Recursive 방식보다 Markdown Header를 기준으로 하나의 기술 개념을 하나의 chunk로 유지하는 방식을 최종 선택했다.

## 2. 기준선에서 무엇을 개선했는가?

BGE-M3를 이용한 Dense Retrieval을 기준선으로 설정했다.

첫 번째 개선에서는 BM25를 추가하여 Dense Retrieval과 lexical retrieval을 함께 사용하는 Hybrid Search를 적용했다. 기술 문서에서는 자연어 의미뿐 아니라 API 이름이나 클래스명처럼 정확한 문자열이 중요한 질문도 존재하기 때문에 두 검색 방식을 결합해 볼 필요가 있다고 판단했다.

두 번째 개선에서는 Hybrid Search의 결과를 RRF로 결합하면서 `k=10, 60, 100, 200`을 비교했다.

각 실험의 실제 Hit@3과 기준선 대비 변화는 위 결과표에 기록하였다.

## 3. 효과가 없었거나 나빠진 실험은 무엇이며 왜 그런 결과가 나왔는가?

E5의 prefix 적용 여부와 embedding normalization을 별도로 실험했다.

이 실험에서 Hit@3 변화가 크지 않았다면, 이번 문서와 평가 질문에서는 해당 설정 변화가 검색 순위를 바꿀 정도의 영향을 주지 않았다고 해석했다.

또한 긴 section을 만들어 silent failure를 확인했다. 하나의 chunk에 너무 많은 내용이 포함되면 embedding 모델이 전체 내용을 하나의 vector로 표현해야 하기 때문에 특정 문장의 중요도가 낮아질 수 있다.

따라서 검색 성능은 embedding 모델만의 문제가 아니라 문서 구조, chunk 크기, 질문 형태가 함께 영향을 미친다는 점을 확인했다.

## 4. 이번 실험을 통해 무엇을 배웠는가?

이번 과제를 통해 RAG에서는 LLM의 생성 능력뿐 아니라 그 앞단의 retrieval 설계가 매우 중요하다는 점을 확인했다.

특히 기술 문서에서는 의미가 비슷한 문서를 찾는 것뿐 아니라 API 이름이나 클래스명처럼 정확한 식별자를 찾는 검색도 필요하다.

또한 모든 실험이 성능 향상으로 이어지는 것은 아니며, 변화가 없었던 실험도 그 원인을 문서·질문·모델의 특성으로 설명할 수 있다면 의미 있는 실험 결과가 될 수 있다는 것을 배웠다.

향후에는 Query Rewriting, Reranker, Parent-Child Chunking 등을 추가하여 검색 품질을 개선하고 실제 Spring 기반 AI 서비스의 retrieval 모듈로 확장해 보고 싶다.

## 10. 프로젝트 요약

- **문서**: Spring AI Retrieval Augmented Generation 공식 문서
- **Embedding**: BGE-M3
- **Vector DB**: Qdrant
- **LLM**: Qwen2.5-7B-Instruct
- **Baseline**: BGE-M3 Dense Retrieval
- **개선 1**: BM25 + Dense Retrieval Hybrid Search
- **개선 2**: RRF k 값 변화 실험
- **추가 실험**: E5 prefix / no-prefix 비교
- **평가 지표**: Hit@3
- **오프라인 검증**: `HF_HUB_OFFLINE=1`, `TRANSFORMERS_OFFLINE=1`
- **Baseline Hit@3**: 86.7%
- **Hybrid Search Hit@3**: 86.7%
- **RRF k=10/60/100/200**: 모두 86.7%
- **E5 prefix**: 80.0%
- **E5 no-prefix**: 86.7%

이번 프로젝트에서는 구조화된 기술 문서를 대상으로 로컬
Embedding → Qdrant → Retrieval → Local LLM으로 이어지는
RAG 파이프라인을 구축하고, 검색 방법과 embedding 설정을 변경하면서
Hit@3를 비교하였다.